<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Ridge Regression

*Session 5 · Notebook 02.05 · Lecture · Student version*

## Overview

**Ordinary least squares (OLS)**, the plain linear regression from the previous notebooks (scikit-learn's `LinearRegression`), fits its coefficients by minimising the sum of squared errors. It can produce huge, unstable coefficients when features are correlated or numerous, which hurts both reliability and interpretation. **Ridge regression** adds a penalty on the size of the coefficients, shrinking them toward zero to make the model more stable and less prone to overfitting. This notebook shows the problem on a small synthetic example, introduces the Ridge penalty and its `alpha` control, applies it to a real housing dataset, visualises how coefficients shrink as `alpha` grows, and chooses `alpha` by cross-validation.

## Learning Objectives

By the end of this notebook you will be able to:

- Explain why correlated or numerous features make ordinary least squares (OLS) unstable.
- Describe the Ridge (L2) penalty and the role of `alpha`.
- Fit Ridge in a scaled workflow and compare its coefficients to ordinary regression.
- Read a coefficient-path plot (shrinkage as `alpha` increases).
- Choose `alpha` with `RidgeCV` (cross-validation).

## Prerequisites

- Session 5 notebooks 02.02 and 02.03 (linear regression, coefficients) and 01.03 (scaling).
- Comfort with `train_test_split` and standardisation.

## Index

1. [Why this matters for risk analysis](#sec1)
2. [What is Ridge regression?](#sec2)
3. [Synthetic data: why plain regression goes unstable](#sec3)
4. [Real data: the housing dataset](#sec4)
5. [Ridge vs ordinary regression](#sec5)
6. [The effect of alpha (coefficient path)](#sec6)
7. [Choosing alpha with cross-validation](#sec7)
8. [Exercises](#exercises)
9. [Challenge](#challenge)
10. [Key Takeaways](#takeaways)
11. [Further Reading](#reading)

<a id="setup"></a>
# Section 0: Setup

We build a small synthetic example first, then use `boston.csv` (housing data), read from the repo-root `datasets/` folder (two levels up).

> **Note on the Boston Housing dataset.** It is a classic teaching set, but it has a well-documented ethical problem: one feature (`B`) is engineered from the proportion of Black residents, reflecting biased assumptions of its era, and scikit-learn removed it for this reason. We use it here only as a faithful teaching example of Ridge; for real work prefer a dataset without such features (the next notebook uses California housing).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV
from sklearn.metrics import mean_squared_error, r2_score

sns.set_theme(style='whitegrid')
np.random.seed(42)
print('Libraries ready.')

<a id="sec1"></a>
# Section 1: Why this matters for risk analysis

Risk models often have **many features**, and those features are frequently **correlated** (income, balance and limit all move together, for example).

| Problem | Consequence | How Ridge helps |
|---|---|---|
| Correlated features | Coefficients become huge and unstable, flipping sign between samples | Shrinks them to stable, sensible values |
| Many features, little data | The model overfits the training set | The penalty curbs overfitting |
| Unstable coefficients | The model cannot be trusted or explained | Stable coefficients are auditable |

Ridge keeps the interpretability of linear regression while making it robust, which is exactly what a production risk model needs.

<a id="sec2"></a>
# Section 2: What is Ridge regression?

**Definition:** Ridge regression is linear regression with an extra **L2 penalty** on the coefficients. It minimises the usual squared error **plus** `alpha` times the sum of the squared coefficients, which pushes the coefficients toward (but not exactly to) zero.

**Example:** with two nearly identical features, ordinary regression might give one a coefficient of +50 and the other -47; Ridge instead gives both a small, shared value.

**Analogy:** a leash on the coefficients. Ordinary regression lets them run as large as they like to fit the data; Ridge holds them back, trading a little training fit for a lot of stability.

**Explanation:**

- **`alpha`** controls the strength: `alpha = 0` is ordinary regression; larger `alpha` means stronger shrinkage (smaller coefficients).
- Ridge shrinks coefficients but keeps them all (none become exactly zero); the next notebook (Lasso) can zero them out for feature selection.
- **Scaling is required**: the penalty treats all coefficients equally, so features must be standardised first, or large-scale features would be penalised unfairly.

**The formula (how the coefficients are found):**

Ridge minimises the squared error **plus** the L2 penalty (the sum of squared coefficients):

$$\text{minimise} \quad \sum_i (y_i - \hat{y}_i)^2 + \alpha \sum_j \beta_j^2$$

Like ordinary regression this has an exact, direct solution, just with $\alpha I$ added to the diagonal before inverting:

$$\boldsymbol{\beta} = (X^\top X + \alpha I)^{-1} X^\top y$$

That extra $\alpha I$ is what shrinks the coefficients and stabilises the inverse (for any $\alpha > 0$ the matrix is always invertible, even when features are correlated). Setting $\alpha = 0$ gives back ordinary least squares. See the side-note notebook `05_18`.

**scikit-learn documentation:** [`Ridge`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html) and [`RidgeCV`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.RidgeCV.html)

<a id="sec3"></a>
# Section 3: Synthetic data - why plain regression goes unstable

We create two features that are almost identical (correlation ~0.9998), where the target really depends on the first. Ordinary regression cannot tell the two apart, so it splits the effect between them arbitrarily, and that split swings wildly from sample to sample. Ridge shares the effect stably.

In [ ]:
rng = np.random.default_rng(1)
n = 60
x1 = rng.normal(0, 1, n)
x2 = x1 + rng.normal(0, 0.02, n)          # a near-duplicate of x1
y = 3 * x1 + 2 + rng.normal(0, 0.4, n)    # the target really depends on x1
X = np.column_stack([x1, x2])
print('Correlation of x1 and x2:', round(np.corrcoef(x1, x2)[0, 1], 4))

ols = LinearRegression().fit(X, y)
ridge = Ridge(alpha=10).fit(X, y)
print('OLS coefficients      :', ols.coef_.round(2), '(arbitrary split)')
print('Ridge coefficients    :', ridge.coef_.round(2), '(shared, shrunk)')

The real test is **stability**. We refit each model on several bootstrap resamples of the data and look at how much the coefficients move. Ordinary regression's coefficients swing enormously; Ridge's barely budge.

In [ ]:
def bootstrap_coef_std(make_model, reps=12):
    coefs = []
    for s in range(reps):
        r = np.random.default_rng(s)
        idx = r.integers(0, n, n)
        coefs.append(make_model().fit(X[idx], y[idx]).coef_)
    return np.array(coefs).std(axis=0)

<a id="sec4"></a>
# Section 4: Real data - the housing dataset

We load the housing data, split it, and standardise the features (fitting the scaler on the training data only). The target is the last column (median home value); the rest are features.

In [ ]:
# Or read directly from the public S3 bucket (no local file needed):
# df = pd.read_csv('https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/Data_Sources_CBS_Risk/Session_5/boston.csv')
# ...or read the paths from a config file (the local read below stays the default):
# from config import session_datasets_http
# df = pd.read_csv(session_datasets_http["boston"])
# Or from S3 with Spark, then to pandas (needs a SparkSession, e.g. on Databricks):
# df = spark.read.csv("s3://rockborne-bucket-01-cbs/Data_Sources_CBS_Risk/Session_5/boston.csv", header=True, inferSchema=True).toPandas()

<a id="sec5"></a>
# Section 5: Ridge vs ordinary regression

We fit an ordinary linear regression and a Ridge (default `alpha=1`) on the same scaled data, compare their test performance, and, most tellingly, compare their coefficients.

In [ ]:
# Your turn. Write your solution here:

In [ ]:
# Compare the coefficients side by side

<a id="sec6"></a>
# Section 6: The effect of alpha (coefficient path)

Plotting each coefficient against `alpha` gives the classic **coefficient path**: as `alpha` increases (more penalty), every coefficient is pulled toward zero. This shows visually how Ridge trades flexibility for stability.

In [ ]:
# Your turn. Write your solution here:

<a id="sec7"></a>
# Section 7: Choosing alpha with cross-validation

Rather than guessing `alpha`, `RidgeCV` tries a list of values and picks the one that performs best under cross-validation on the training data. This is the principled way to set the penalty strength. (Cross-validation itself is introduced in notebook 11.)

In [ ]:
# Your turn. Write your solution here:

<a id="exercises"></a>
# Section 8: Exercises

### Exercise 1: A stronger penalty

Fit a Ridge with `alpha=100` on the scaled training data and report its test R-squared. How does the very strong penalty affect performance compared with `alpha=1`?

In [ ]:
# Your turn. Write your solution here:


### Exercise 2: Coefficient magnitude vs alpha

For alphas 0.1, 1, 10 and 100, print the sum of squared Ridge coefficients (fit on the scaled training data). Confirm it falls as alpha rises.

In [ ]:
# Your turn. Write your solution here:


### Exercise 3: Which features survive shrinkage?

Using the tuned `ridge_cv` model, print the three features with the largest absolute coefficients (the drivers the penalised model still relies on most).

In [ ]:
# Your turn. Write your solution here:


<a id="challenge"></a>
## Challenge (optional): does Ridge beat ordinary regression here?

Compare ordinary linear regression against the cross-validated Ridge fairly, using 5-fold cross-validation on the whole (scaled) dataset. Compute the mean cross-validated R-squared for each and comment on whether the penalty helped on this dataset (it does not always, and that is a useful lesson).

In [ ]:
# Your turn. Write your solution here:


<a id="takeaways"></a>
## Key Takeaways

| Concept / command | What it does |
|---|---|
| Ridge regression | Linear regression + an L2 penalty on coefficient size |
| `alpha` | Penalty strength: 0 = ordinary regression, larger = more shrinkage |
| Shrinkage | Coefficients pulled toward 0 (but never exactly 0) |
| Stability under collinearity | Ridge's main benefit vs ordinary regression |
| Scaling required | Standardise first so the penalty is fair across features |
| `Ridge(alpha=...)` | Fit a Ridge model |
| coefficient path | Coefficients vs alpha; shows shrinkage visually |
| `RidgeCV` | Choose alpha by cross-validation |


## Conclusion

You can now explain why ordinary regression is fragile with correlated or numerous features, use Ridge to stabilise it, read the coefficient path, and choose the penalty strength by cross-validation. The next notebook covers Lasso, which shrinks some coefficients all the way to zero and so performs automatic feature selection.

<a id="reading"></a>
## Further Reading & Resources

- [scikit-learn: Ridge regression](https://scikit-learn.org/stable/modules/linear_model.html#ridge-regression-and-classification) the L2 penalty and RidgeCV.
- [Regularisation explained](https://scikit-learn.org/stable/auto_examples/linear_model/plot_ridge_path.html) the coefficient-path example.
- [On the Boston dataset's issues](https://scikit-learn.org/1.0/modules/generated/sklearn.datasets.load_boston.html) why it was deprecated.